In [1]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM , AutoTokenizer , GenerationConfig , TrainingArguments , Trainer
import torch
import time
import evaluate
import pandas as pd
import numpy as np
from dotenv import load_dotenv

W1021 23:10:14.378000 24084 venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
load_dotenv()

True

In [3]:
huggingface_dataset_name="knkarthick/dialogsum"

dataset=load_dataset(huggingface_dataset_name)

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [5]:
dataset.save_to_disk("./dataset")

Saving the dataset (0/1 shards):   0%|          | 0/12460 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1500 [00:00<?, ? examples/s]

## Loading original/base model

In [6]:
model_name="google/flan-t5-base"

original_model=AutoModelForSeq2SeqLM.from_pretrained(pretrained_model_name_or_path=model_name,dtype=torch.bfloat16).to("cuda")

tokenizer=AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [7]:
print("Model dtype:", next(original_model.parameters()).dtype)
print("CUDA available:", torch.cuda.is_available())

Model dtype: torch.bfloat16
CUDA available: True


In [8]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params=0
    all_model_params=0
    for _,param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"Trainable model parameters : {trainable_model_params}\nAll model parameters: {all_model_params}\nPercentage of trainable model parameters {(trainable_model_params / all_model_params)*100}"

In [9]:
print(print_number_of_trainable_model_parameters(original_model))

Trainable model parameters : 247577856
All model parameters: 247577856
Percentage of trainable model parameters 100.0


In [10]:
index=200

dialogue=dataset['test'][index]['dialogue']
summary=dataset['test'][index]['summary']

prompt= f"""
        Summarize the following conversation
        {dialogue}

        Summary:
        """
inputs=tokenizer(prompt,return_tensors="pt")
output=tokenizer.decode(
    original_model.generate(
        inputs['input_ids'].to("cuda"),
        max_new_tokens=200
    )[0],
    skip_special_tokens=True
)

dash_line="-".join("" for i in range(100))

print(dash_line)
print(f"Input prompt:\n {prompt}")
print(dash_line)
print(f"Baseline Human summary:\n{summary}")
print(dash_line)
print(f"Model generation - ZERO SHOT:\n {output}")

---------------------------------------------------------------------------------------------------
Input prompt:
 
        Summarize the following conversation
        #Person1#: Have you considered upgrading your system?
#Person2#: Yes, but I'm not sure what exactly I would need.
#Person1#: You could consider adding a painting program to your software. It would allow you to make up your own flyers and banners for advertising.
#Person2#: That would be a definite bonus.
#Person1#: You might also want to upgrade your hardware because it is pretty outdated now.
#Person2#: How can we do that?
#Person1#: You'd probably need a faster processor, to begin with. And you also need a more powerful hard disc, more memory and a faster modem. Do you have a CD-ROM drive?
#Person2#: No.
#Person1#: Then you might want to add a CD-ROM drive too, because most new software programs are coming out on Cds.
#Person2#: That sounds great. Thanks.

        Summary:
        
------------------------------------

In [11]:
def tokenize_function(example):
    start_prompt="Summarize the following conversation.\n\n"
    end_prompt="\n\nSummary"
    prompt=[start_prompt+dialogue+end_prompt for dialogue in example["dialogue"]]
    example["input_ids"]=tokenizer(prompt, padding="max_length",truncation=True,return_tensors="pt").input_ids
    example["labels"]=tokenizer(example["summary"],padding="max_length",truncation=True,return_tensors="pt").input_ids
    return example

In [12]:
tokenized_datasets=dataset.map(tokenize_function,batched=True)
tokenized_datasets=tokenized_datasets.remove_columns(["id","topic","dialogue","summary"])

In [13]:
print(f"Shapes of the datasets:")
print(f"Training: {tokenized_datasets['train'].shape}")
print(f"Validation: {tokenized_datasets['validation'].shape}")
print(f"Test: {tokenized_datasets['test'].shape}")

Shapes of the datasets:
Training: (12460, 2)
Validation: (500, 2)
Test: (1500, 2)


In [14]:
tokenized_datasets=tokenized_datasets.filter(lambda example,index: index % 100 == 0,with_indices=True)

In [15]:
print(f"Shapes of the datasets:")
print(f"Training: {tokenized_datasets['train'].shape}")
print(f"Validation: {tokenized_datasets['validation'].shape}")
print(f"Test: {tokenized_datasets['test'].shape}")

Shapes of the datasets:
Training: (125, 2)
Validation: (5, 2)
Test: (15, 2)


In [16]:
print(tokenized_datasets)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 125
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 5
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 15
    })
})


## Fine tuning base model on dialogsum dataset and only on 1 percent of dataset

In [17]:
output_dir=f"../data/dialogue-summary-training-{str(int(time.time()))}"

training_args=TrainingArguments(
    output_dir=output_dir,
    learning_rate=1e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    use_cpu=False,
    per_device_train_batch_size=1,
    bf16=True,
    fp16=False
)

trainer=Trainer(
    model=original_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"]
)

In [18]:
trainer.train()

Step,Training Loss
100,48.001304
200,46.459995
300,45.760752


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=375, training_loss=46.44151236979167, metrics={'train_runtime': 84.5274, 'train_samples_per_second': 4.436, 'train_steps_per_second': 4.436, 'total_flos': 256784007168000.0, 'train_loss': 46.44151236979167, 'epoch': 3.0})

In [19]:
trainer.save_model("../models/flan-t5-dialogsum")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
instruct_model=AutoModelForSeq2SeqLM.from_pretrained("../models/flan-t5-dialogsum").to("cuda")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [21]:
print("Original model:", next(original_model.parameters()).device)
print("Fine-tuned model:", next(instruct_model.parameters()).device)

Original model: cuda:0
Fine-tuned model: cuda:0


In [22]:
dialogue=dataset['test'][index]['dialogue']
human_baseline_summary=dataset['test'][index]['summary']

prompt=f"""
        Summarize the following conversation
        {dialogue}
        Summary:
"""
input_ids=tokenizer(prompt,return_tensors="pt").input_ids
input_ids=input_ids.to(original_model.device)
original_model_outputs=original_model.generate(input_ids=input_ids,generation_config=GenerationConfig(max_new_tokens=200,num_beams=1))
original_model_text_output=tokenizer.decode(original_model_outputs[0],skip_special_tokens=True)

instruct_model_outputs=instruct_model.generate(input_ids=input_ids,generation_config=GenerationConfig(max_new_tokens=200,num_beams=1))

instruct_model_text_output=tokenizer.decode(instruct_model_outputs[0],skip_special_tokens=True)

## Comparing base model and fine-tuned model

In [23]:
print(dash_line)
print(f"HUMAN BASELINE SUMMARY \n{human_baseline_summary}")
print(dash_line)
print(f"ORIGINAL MODEL \n{original_model_text_output}")
print(dash_line)
print(f"FINE TUNED MODEL\n{instruct_model_text_output}")

---------------------------------------------------------------------------------------------------
HUMAN BASELINE SUMMARY 
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.
---------------------------------------------------------------------------------------------------
ORIGINAL MODEL 
#Person2#: Yes, I'm thinking of upgrading my system.
---------------------------------------------------------------------------------------------------
FINE TUNED MODEL
#Person1#: I'm thinking of upgrading my computer.


## Evaluation

In [24]:
rouge=evaluate.load("rouge")

In [25]:
dialogues=dataset['test'][0:10]['dialogue']
human_baseline_summaries=dataset['test'][0:10]['summary']

In [26]:
original_model_summaries=[]
instruct_model_summaries=[]

for _,dialogue in enumerate(dialogues):
    prompt=f"""
            Summarize the following conversation
            {dialogue}
            Summary:
            """
    input_ids=tokenizer(prompt,return_tensors="pt").input_ids
    input_ids=input_ids.to(original_model.device)
    original_model_outputs=original_model.generate(input_ids=input_ids,generation_config=GenerationConfig(max_new_tokens=200))
    original_model_text_output=tokenizer.decode(original_model_outputs[0],skip_special_tokens=True)
    original_model_summaries.append(original_model_text_output)

    instruct_model_outputs=instruct_model.generate(input_ids=input_ids,generation_config=GenerationConfig(max_new_tokens=200))
    instruct_model_text_output=tokenizer.decode(instruct_model_outputs[0],skip_special_tokens=True)
    instruct_model_summaries.append(instruct_model_text_output)

zipped_summaries=list(zip(human_baseline_summaries,original_model_summaries,instruct_model_summaries))

df=pd.DataFrame(zipped_summaries,columns=["HUMAN BASELINE SUMMARY","ORIGINAL MODEL SUMMARY","INSTRUCT MODEL SUMMARY"])

In [27]:

df

,HUMAN BASELINE SUMMARY,ORIGINAL MODEL SUMMARY,INSTRUCT MODEL SUMMARY
0,Ms. Dawson helps #Person1# to write a memo to ...,This memo should go to all employees by this a...,The memo will go out to all employees by this ...
1,In order to prevent employees from wasting tim...,This memo is to be distributed to all employee...,The memo will go out to all employees by this ...
2,Ms. Dawson takes a dictation for #Person1# abo...,"#Person1#: Yes, sir... #Person2#: Yes, sir... ...",The memo will go out to all employees by this ...
3,#Person2# arrives late because of traffic jam....,The conversation focuses on the following topics:,The traffic jam at the Carrefour intersection ...
4,#Person2# decides to follow #Person1#'s sugges...,The following people are discussing ways to ge...,The traffic jam at the Carrefour intersection ...
5,#Person2# complains to #Person1# about the tra...,The conversation started with a question about...,The traffic jam at the Carrefour intersection ...
6,#Person1# tells Kate that Masha and Hero get d...,Masha and Hero are getting divorced.,Masha and Hero are getting divorced.
7,#Person1# tells Kate that Masha and Hero are g...,Masha and Hero are getting divorced.,Masha and Hero are getting divorced.
8,#Person1# and Kate talk about the divorce betw...,Masha and Hero are getting divorced.,Masha and Hero are getting divorced.
9,#Person1# and Brian are at the birthday party ...,Brian's birthday is today and he wants to cele...,"#Person1#: Happy Birthday, Brian. #Person2#: T..."


In [28]:
original_model_results=rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries[0:len(original_model_summaries)],
    use_aggregator=True,
    use_stemmer=True
)
instruct_model_results=rouge.compute(
    predictions=instruct_model_summaries,
    references=human_baseline_summaries[0:len(instruct_model_summaries)],
    use_aggregator=True,
    use_stemmer=True
)

In [29]:
print(dash_line)
print(f"ORIGINAL MODEL RESULTS")
print(original_model_results)
print(dash_line)
print(f"INSTRUCT MODEL RESULTS")
print(instruct_model_results)
print(dash_line)

---------------------------------------------------------------------------------------------------
ORIGINAL MODEL RESULTS
{'rouge1': np.float64(0.26452805475695673), 'rouge2': np.float64(0.11075362318840579), 'rougeL': np.float64(0.22232360328208786), 'rougeLsum': np.float64(0.22748683798025898)}
---------------------------------------------------------------------------------------------------
INSTRUCT MODEL RESULTS
{'rouge1': np.float64(0.24226965394938516), 'rouge2': np.float64(0.10814756258234519), 'rougeL': np.float64(0.208376683662238), 'rougeLsum': np.float64(0.2111736059552409)}
---------------------------------------------------------------------------------------------------


In [30]:
print("Absolute percentage improvement of INSTRUCT MODEL over HUMAN BASELINE")

improvement=(np.array(list(instruct_model_results.values()))-np.array(list(original_model_results.values())))
for key,value in zip(instruct_model_results.keys(),improvement):
    print(f"{key}:{value*100:.2f}%")

Absolute percentage improvement of INSTRUCT MODEL over HUMAN BASELINE
rouge1:-2.23%
rouge2:-0.26%
rougeL:-1.39%
rougeLsum:-1.63%


## PEFT (PARAMETER EFFICIENT FINE-TUNING) LoRA

In [31]:
from peft import LoraConfig,get_peft_model,TaskType

In [32]:
lora_config=LoraConfig(
    r=32, #Rank
    lora_alpha=32,
    target_modules=["q","v"],
    lora_dropout=0.5,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM #FLAN-T5
)

In [33]:
peft_model=get_peft_model(original_model,lora_config)

In [34]:
print(print_number_of_trainable_model_parameters(peft_model))

Trainable model parameters : 3538944
All model parameters: 251116800
Percentage of trainable model parameters 1.4092820552029972


In [35]:
output_dir=f"../data/peft-dialogue-summary-training-{str(int(time.time()))}"

peft_training_args=TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    auto_find_batch_size=True,
    learning_rate=1e-5,
    logging_steps=100,
    use_cpu=False,
    bf16=True,
    fp16=False,
)

peft_trainer=Trainer(
    model=peft_model,
    args=peft_training_args,
    train_dataset=tokenized_datasets["train"]
)

In [36]:
peft_trainer.train()

Step,Training Loss


TrainOutput(global_step=16, training_loss=44.95854568481445, metrics={'train_runtime': 243.6922, 'train_samples_per_second': 0.513, 'train_steps_per_second': 0.066, 'total_flos': 86953623552000.0, 'train_loss': 44.95854568481445, 'epoch': 1.0})

In [37]:
peft_model_path=f"../models/flan-t5-peft"

In [38]:
peft_trainer.save_model(peft_model_path)

In [39]:
from peft import PeftConfig,PeftModel

In [40]:
peft_model_base=AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base",dtype=torch.bfloat16)
tokenizer=AutoTokenizer.from_pretrained("google/flan-t5-base")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [41]:
peft_model=PeftModel.from_pretrained(peft_model_base,peft_model_path,dtype=torch.bfloat16,is_trainable=False).to("cuda")

In [42]:
print(print_number_of_trainable_model_parameters(peft_model))

Trainable model parameters : 0
All model parameters: 251116800
Percentage of trainable model parameters 0.0


In [43]:
dialogue=dataset["test"][index]["dialogue"]
summary=dataset["test"][index]["summary"]

prompt=f"""
        Summarize the following conversation
        {dialogue}

        Summary:
        """
input_ids=tokenizer(prompt,return_tensors="pt").input_ids
input_ids=input_ids.to(original_model.device)
original_model_outputs=original_model.generate(input_ids,generation_config=GenerationConfig(max_new_tokens=200,num_beams=1))
original_model_text_output=tokenizer.decode(original_model_outputs[0],skip_special_tokens=True)

instruct_model_outputs=instruct_model.generate(input_ids,generation_config=GenerationConfig(max_new_tokens=200,num_beams=1))
instruct_model_text_output=tokenizer.decode(instruct_model_outputs[0],skip_special_tokens=True)

peft_model_outputs=peft_model.generate(input_ids=input_ids,generation_config=GenerationConfig(max_new_tokens=200,num_beams=1))
peft_model_text_output=tokenizer.decode(peft_model_outputs[0],skip_special_tokens=True)

In [44]:
print(dash_line)
print(f"HUMAN BASELINE SUMMARY")
print(human_baseline_summary)
print(dash_line)
print(f"ORIGINAL MODEL")
print(original_model_text_output)
print(dash_line)
print(f"INSTRUCT MODEL")
print(instruct_model_text_output)
print(dash_line)
print(f"PEFT MODEL")
print(peft_model_text_output)
print(dash_line)

---------------------------------------------------------------------------------------------------
HUMAN BASELINE SUMMARY
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.
---------------------------------------------------------------------------------------------------
ORIGINAL MODEL
#Person1#: I'd like to upgrade my software. #Person1#: I'd like to add a painting program to my software.
---------------------------------------------------------------------------------------------------
INSTRUCT MODEL
#Person1#: I'm thinking of upgrading my computer.
---------------------------------------------------------------------------------------------------
PEFT MODEL
#Person1#: I'm thinking of upgrading my computer.
---------------------------------------------------------------------------------------------------


In [45]:
dialogues=dataset['test'][0:9]['dialogue']
human_baseline_summaries=dataset['test'][0:9]['summary']

original_model_summaries=[]
instruct_model_summaries=[]
peft_model_summaries=[]
for idx,dialogue in enumerate(dialogues):
    prompt=f"""
            Summarize the following conversation

            {dialogue}
            Summary:
            """
    input_ids= tokenizer(prompt,return_tensors="pt").input_ids
    input_ids=input_ids.to("cuda")
    human_baseline_text_output=human_baseline_summaries[idx]

    original_model_outputs=original_model.generate(input_ids=input_ids,generation_config=GenerationConfig(max_new_tokens=200,num_beams=1))
    original_model_text_output=tokenizer.decode(original_model_outputs[0],skip_special_tokens=True)

    instruct_model_outputs=instruct_model.generate(input_ids=input_ids,generation_config=GenerationConfig(max_new_tokens=200,num_beams=1))
    instruct_model_text_output=tokenizer.decode(instruct_model_outputs[0],skip_special_tokens=True)

    peft_model_outputs=peft_model.generate(input_ids=input_ids,generation_config=GenerationConfig(max_new_tokens=200,num_beams=1))
    peft_model_text_output=tokenizer.decode(peft_model_outputs[0],skip_special_tokens=True)

    original_model_summaries.append(original_model_text_output)
    instruct_model_summaries.append(instruct_model_text_output)
    peft_model_summaries.append(peft_model_text_output)

zipped_summaries=list(zip(human_baseline_summaries,original_model_summaries,instruct_model_summaries,peft_model_summaries))

df=pd.DataFrame(zipped_summaries,columns=["HUMAN BASELINE SUMMARY","ORIGINAL MODEL SUMMARY","INSTRUCT MODEL SUMMARY","PEFT MODEL SUMMARY"])


In [46]:
df

,HUMAN BASELINE SUMMARY,ORIGINAL MODEL SUMMARY,INSTRUCT MODEL SUMMARY,PEFT MODEL SUMMARY
0,Ms. Dawson helps #Person1# to write a memo to ...,Employees are required to take dictamens and m...,The memo will go out to all employees by this ...,The memo will go out to all employees by this ...
1,In order to prevent employees from wasting tim...,The memo is to be distributed to all employees...,The memo will go out to all employees by this ...,The memo will go out to all employees by this ...
2,Ms. Dawson takes a dictation for #Person1# abo...,Employees will be asked to change their commun...,The memo will go out to all employees by this ...,The memo will go out to all employees by this ...
3,#Person2# arrives late because of traffic jam....,The traffic jam is a problem for Person 1.,The traffic jam at the Carrefour intersection ...,The traffic jam at the Carrefour intersection ...
4,#Person2# decides to follow #Person1#'s sugges...,The traffic jam isn't a problem anymore for th...,The traffic jam at the Carrefour intersection ...,The traffic jam at the Carrefour intersection ...
5,#Person2# complains to #Person1# about the tra...,The traffic is getting worse.,The traffic jam at the Carrefour intersection ...,The traffic jam at the Carrefour intersection ...
6,#Person1# tells Kate that Masha and Hero get d...,Masha and Hero are divorced.,Masha and Hero are getting divorced.,Masha and Hero are getting divorced.
7,#Person1# tells Kate that Masha and Hero are g...,#Person1#: I'm not sure what happened to Masha...,Masha and Hero are getting divorced.,Masha and Hero are getting divorced.
8,#Person1# and Kate talk about the divorce betw...,"#Person1#: #Person2#: Well, I don't really kno...",Masha and Hero are getting divorced.,Masha and Hero are getting divorced.


In [47]:
rouge=evaluate.load('rouge')

original_model_results=rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries[0:len(original_model_summaries)],
    use_aggregator=True,
    use_stemmer=True
)

instruct_model_results=rouge.compute(
    predictions=instruct_model_summaries,
    references=human_baseline_summaries[0:len(instruct_model_summaries)],
    use_aggregator=True,
    use_stemmer=True
)

peft_model_results=rouge.compute(
    predictions=peft_model_summaries,
    references=human_baseline_summaries[0:len(peft_model_summaries)],
    use_aggregator=True,
    use_stemmer=True
)

In [48]:
print(dash_line)
print(original_model_results)
print(dash_line)
print(instruct_model_results)
print(dash_line)
print(peft_model_results)

---------------------------------------------------------------------------------------------------
{'rouge1': np.float64(0.2414608343608976), 'rouge2': np.float64(0.10161686833351001), 'rougeL': np.float64(0.19717531174520422), 'rougeLsum': np.float64(0.1993796561744063)}
---------------------------------------------------------------------------------------------------
{'rouge1': np.float64(0.262736191269226), 'rouge2': np.float64(0.12178012004098962), 'rougeL': np.float64(0.22398348014871594), 'rougeLsum': np.float64(0.22801242462807725)}
---------------------------------------------------------------------------------------------------
{'rouge1': np.float64(0.262736191269226), 'rouge2': np.float64(0.12178012004098962), 'rougeL': np.float64(0.22398348014871594), 'rougeLsum': np.float64(0.22801242462807725)}


In [49]:
print("Absolute percentage improvement of PEFT MODEL over HUMAN BASELINE")

improvement=(np.array(list(peft_model_results.values()))-np.array(list(original_model_results.values())))
for key,value in zip(instruct_model_results.keys(),improvement):
    print(f"{key}:{value*100:.2f}%")

Absolute percentage improvement of PEFT MODEL over HUMAN BASELINE
rouge1:2.13%
rouge2:2.02%
rougeL:2.68%
rougeLsum:2.86%
